In [426]:
from collections import defaultdict

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.dagcircuit import DAGCircuit, DAGOpNode, DAGInNode, DAGOutNode
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.transpiler.layout import Layout
from qiskit.circuit.quantumregister import Qubit

import matplotlib.pyplot as plt
import networkx as nx

from vfsexp import Vf

In [427]:
def extract_circuit(filename, path, node_num):
    # compose the input Quantum Circuit
    q = QuantumRegister(node_num, 'q')
    qc = QuantumCircuit(q)
    cir_temp = qc.from_qasm_file(path+filename)
    qc.compose(cir_temp, inplace=True)
    if node_num < cir_temp.num_qubits:
        raise Exception("Cannot compose circuit!") 

    return qc

""" From a QuantumCircuit, remove all 1qb gates and any 2qb gate [p,q] that has a parent [p,q] or [q,p]"""
def remove_1q_and_consecutive_2q_gates_in_circuit(circuit):

    newcirc = QuantumCircuit(circuit.num_qubits)
    record = [] #the current end layer of the reduced circuit
    for instruction in circuit:
        if len(instruction.qubits) != 2: continue
        if instruction.operation.name == 'cx': # only consider cx gates
            p, q = instruction.qubits[0]._index, instruction.qubits[1]._index
            if [p,q] in record or [q,p] in record:
                continue
            for x in record:
                if p in x or q in x:
                    record.remove(x)
            record.append([p,q])            
            newcirc.cx(p,q)
    return newcirc

def graph_of_circuit(circuit):
    ''' Return the induced graph of the reduced circuit C
            - node set: qubits in C
            - edge set: all pair (p,q) if CNOT [p,q] or CNOT[q,p] in C
        Args:
            C (list): the input reduced circuit
        Returns:
            g (Graph)
    '''   
    g = nx.Graph()
    for gate in circuit:
        if gate.operation.name != 'cx': continue
        p, q = gate.qubits[0]._index, gate.qubits[1]._index
        g.add_edge(p,q)
    return g  

def has_cycle_3(graph):
    for u in graph.nodes():
        for v in graph.neighbors(u):
            for w in graph.neighbors(v):
                if u != w and graph.has_edge(u, w):
                    return True
    return False

def fast_no_embedding(subgraph, graph):
    # for AG with no 3-cycle
    if has_cycle_3(subgraph) and not has_cycle_3(graph): 
        return True
    
    subgraph_deg = list(nx.degree(subgraph, v) for v in subgraph.nodes())
    subgraph_deg.sort(reverse=True)
    
    graph_deg = list(nx.degree(subgraph, v) for v in subgraph.nodes())
    graph_deg.sort(reverse=True)
    
    l = len(subgraph_deg)
    for idx in range(l):
        if subgraph_deg[idx] > graph_deg[idx]:
            return True
        
    return False

def is_embeddable(g, H, stop):
    '''check if a small graph g is embeddable in a large H, anchor is bool
        g, H (Graph)
        stop (float): time limit for vf2
    '''
    if fast_no_embedding(g, H):
        return False, {}
    
    lng = len(nx.nodes(g))
        
    new_g = g.copy()
    #print(len(new_g.edges()) == len(g.edges()))

    new_H = H.copy()
    temp_map = dict()

    vf2 = Vf(new_g, new_H, {}, stop)
    result = vf2.dfsMatch({})
    if len(result) == lng:
        return True, temp_map.update(result)  
    return False, {}

In [428]:
def _successors(node, dag):
    for _, successor, edge_data in dag.edges(node):
        if not isinstance(successor, DAGOpNode):
            continue
        #"""Our dag is derived from special circuit with 1q gates and 2q consecutive identical gates removed."""
        #if isinstance(node, DAGOpNode) and successor.qargs == node.qargs: 
        #    continue #sl sometimes successor and node are identical, don't know why
        if isinstance(edge_data, Qubit):
            yield successor

In [446]:
from ag import qgrid
import time
import os

path = '../bench/qiskit_circuit_benchmark/' 
filename = 'hidden_linear_6.qasm'

AG = qgrid(2,3)
architecture = qgrid(2,3)
#print(architecture.edges())
#g = nx.Graph()
#g.add_edges_from([(4, 5), (4, 3), (5, 3)])
#print(is_embeddable(g,AG,100))

qc = extract_circuit(filename, path, len(AG.nodes()))
#qc.draw('mpl')

newcirc = remove_1q_and_consecutive_2q_gates_in_circuit(qc)
#for x in newcirc:
#    print(x.qubits[0]._index,x.qubits[1]._index)
#newcirc.draw('mpl')

dag = circuit_to_dag(newcirc)
#for _, input_node in dag.input_map.items(): 
#    print(type(input_node), input_node.wire)
print(dag.count_ops())
    
for node in dag.nodes():
    if type(node) == DAGInNode or type(node) == DAGOutNode:
        continue
        #print(type(node), node.wire._index)
    else:
        #print(type(node), node.qargs[0]._index, node.qargs[1]._index)
        if node.name == 'cx':
            print(node in dag.nodes())
            #dag.remove_op_node(node)
            print(node in dag.nodes())
            break
        
print(dag.count_ops())
        
    
    #print([(successor.qargs[0]._index, successor.qargs[1]._index) for successor in _successors(node, dag)])

{'cx': 8}
True
True
{'cx': 8}


In [449]:
"""Partition a dagCircuit into sections so that each section is embeddable into a given architecture""" 
def partition(dag, architecture, method='greedy'):
    part_list = []
    
    canonical_register = dag.qregs["q"]
    
    print(f'The dagCircuit has {dag.count_ops()} cx gates')

    #print(f'DAG has {len(dag.nodes())} nodes')
    front_layer = dag.front_layer() #sl This front layer contains 1qbg
    for node in front_layer:
        if len(node.qargs) != 2 or node.op.name != 'cx': 
            raise Exception(f'check why dag still has single-qubit gates {type(node), node.op.name}')
    
    # Create a defaultdict with a default value of 0
    applied_predecessors = defaultdict(int)
    for _, input_node in dag.input_map.items(): 
        for successor in _successors(input_node, dag):
            applied_predecessors[successor] += 1

    current_part_dag = dag.copy_empty_like()
    # generate the current layer and examine for each node of the current layer
    
    step = 0
    processed_node = []
    while front_layer and step < 10:
        step += 1
        #print(f'step {step} front layer: \n  {[(node.qargs[0]._index,node.qargs[1]._index) for node in front_layer]}')
        
        execute_gate_list = []
        
        for node in front_layer:
            
            v0, v1 = node.qargs
            
            #print(f'node {v0._index,v1._index}')
            
            # add node to current part
            #current_part_dag.apply_operation_back(node.op, node.qargs, node.cargs)
            #print(f'#gate number {current_part_dag.count_ops()}')
            #print(f'{[(node.qargs[0]._index,node.qargs[1]._index) for node in current_part_dag.nodes() if isinstance(node, DAGOpNode)]}')
            
            temp_qc = dag_to_circuit(current_part_dag)
            temp_graph = graph_of_circuit(temp_qc)
            temp_graph.add_edge(v0._index,v1._index)
            
            EMBEDDABLE = is_embeddable(temp_graph, architecture, stop=10)[0]
            #print(f'node {v0._index,v1._index} {EMBEDDABLE}')

            if EMBEDDABLE:
                execute_gate_list.append(node)
                processed_node.append(node)
                current_part_dag.apply_operation_back(node.op, node.qargs, node.cargs)

                #print(f'yes! {[edge for edge in temp_graph.edges()]}')
            #else:
                # if not embeddable, put the node back 
            #    print(f'temp_graph embeddable? {EMBEDDABLE}! {[edge for edge in temp_graph.edges()]} ')
                
            #    """As the temp_graph is not embeddable, we remove the node from current_part_dag"""
                #print(node in dag.nodes(), current_part_dag.count_ops())
            #    current_part_dag.remove_op_node(node)
                #print(node in current_part_dag.nodes(), current_part_dag.count_ops())
                
            #    x_qc = dag_to_circuit(current_part_dag)
                #print(x_qc)
                #x_qc.draw()
            #    x_graph = graph_of_circuit(x_qc)
            #    E = is_embeddable(x_graph, architecture, stop=10)[0]
            #    print(f'xgraph embeddable? {E}! {[edge for edge in x_graph.edges()]}')

        """The whole layer has been examined!"""
        print(f'step {step} execute gate list: \n  {[(node.qargs[0]._index,node.qargs[1]._index) for node in execute_gate_list]}')

        if execute_gate_list: 
            """Update the front layer!"""
            for node in execute_gate_list:
                front_layer.remove(node)
                #print(f'successors: {[(successor.qargs[0]._index, successor.qargs[1]._index) for successor in _successors(node, dag)]}')
                for successor in _successors(node, dag):
                    applied_predecessors[successor] += 1
                    if applied_predecessors[successor] == len(node.qargs):
                        front_layer.append(successor)
            
        else: 
            """No gate in the current front layer can be added. A new section should be started."""
            part_list.append(current_part_dag)

            # print the corresponding graph edges
            t_qc = dag_to_circuit(current_part_dag)
            t_graph = graph_of_circuit(t_qc)
            E = is_embeddable(t_graph, architecture, stop=10)[0]
            
            print(f'step {step}: {E} construted a section! \n {[edge for edge in t_graph.edges()]}')
            
            """Start a new part dag!"""
            current_part_dag = dag.copy_empty_like()
            print(f'step {step}: start a new part dag {current_part_dag.count_ops()}')

        #print(f'step {step}E front layer: \n  {[(node.qargs[0]._index,node.qargs[1]._index) for node in front_layer]}')

    if len(processed_node) < dag.count_ops()['cx']:
        raise Exception (f'The partition is incomplete! {len(processed_node), dag.count_ops()}')
        
    print(f'{len(processed_node)} nodes have been processed \n {[(node.qargs[0]._index,node.qargs[1]._index) for node in processed_node]}')

    print('The partition is complete!')
    return part_list

In [450]:
X = partition(dag, architecture, method='greedy')

The dagCircuit has {'cx': 8} cx gates
step 1 execute gate list: 
  [(0, 1)]
step 2 execute gate list: 
  [(0, 3), (1, 2)]
step 3 execute gate list: 
  [(1, 4), (2, 3)]
step 4 execute gate list: 
  [(2, 5)]
step 5 execute gate list: 
  []
step 5: True construted a section! 
 [(0, 1), (0, 3), (1, 2), (1, 4), (2, 3), (2, 5)]
step 5: start a new part dag {}
step 6 execute gate list: 
  [(3, 4)]
step 7 execute gate list: 
  [(4, 5)]
8 nodes have been processed 
 [(0, 1), (0, 3), (1, 2), (1, 4), (2, 3), (2, 5), (3, 4), (4, 5)]
The partition is complete!
